# 03 — Model Comparison

**Research:** Early-Stage CKD Detection Using Hybrid Optimization (SGPO v2)

**Team:** Muhammed Abdel-Hamid | Abdul Rahman Al-Bili

**Supervisor:** Dr. Ibrahim | New Mansoura University | CSE015

---

Compare SGPO v2 against standard classifiers to contextualize the optimization results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
plt.style.use('seaborn-v0_8-whitegrid')
print('Ready.')

## 1. Load Comparison Results

In [ ]:
with open('../results/model_comparison_results.json') as f:
    data = json.load(f)

models = data['models']
df = pd.DataFrame(models)

# Display table
display_cols = ['model', 'n_features', 'auc_mean', 'auc_std', 'sensitivity_mean', 'sensitivity_std', 'accuracy_mean', 'f1_mean']
print(df[display_cols].to_string(index=False))

## 2. Performance Comparison Chart

In [ ]:
names = [m['model'] for m in models]
short_names = [n.replace(' (baseline)', '\n(baseline)').replace(' (optimized RF)', '\n(optimized)') for n in names]

metrics = {
    'AUC-ROC': [m['auc_mean'] for m in models],
    'Sensitivity': [m['sensitivity_mean'] for m in models],
    'F1-Score': [m['f1_mean'] for m in models],
}

x = np.arange(len(names))
width = 0.25
colors = ['#2563EB', '#DC2626', '#059669']

fig, ax = plt.subplots(figsize=(10, 6))
for idx, (mname, vals) in enumerate(metrics.items()):
    ax.bar(x + (idx-1)*width, vals, width, label=mname, color=colors[idx], alpha=0.85)

ax.set_ylim(0.85, 0.98)
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=10)
ax.set_ylabel('Score')
ax.set_title('Model Comparison (10-Fold CV)', fontweight='bold', fontsize=14)
ax.legend()

for i, m in enumerate(models):
    ax.text(i, 0.855, f"{m['n_features']} feat.", ha='center', fontsize=9, fontstyle='italic', color='gray')

plt.tight_layout()
plt.show()

## 3. ROC Curve Comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except:
    HAS_XGB = False

df_data = pd.read_csv('../data/processed/mimic_ckd_dataset_final.csv')
X_all = df_data.drop(columns=['subject_id', 'ckd_label'])
y = df_data['ckd_label']

with open('../results/sgpo_v2_results.json') as f:
    sgpo = json.load(f)
sgpo_feats = sgpo['optimization_results']['selected_features']
sgpo_hp = sgpo['optimization_results']['best_hyperparameters']

X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42, stratify=y)

model_specs = [
    ('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42), X_train, X_test),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), X_train, X_test),
]
if HAS_XGB:
    model_specs.append(('XGBoost', XGBClassifier(n_estimators=200, max_depth=8, random_state=42, verbosity=0), X_train, X_test))
model_specs.append(('SGPO v2', RandomForestClassifier(
    n_estimators=sgpo_hp['n_estimators'], max_depth=sgpo_hp['max_depth'],
    min_samples_split=sgpo_hp['min_samples_split'], min_samples_leaf=sgpo_hp['min_samples_leaf'],
    random_state=42, n_jobs=-1), X_train[sgpo_feats], X_test[sgpo_feats]))

colors = ['#6B7280', '#EA580C', '#059669', '#2563EB']

fig, ax = plt.subplots(figsize=(8, 7))
for idx, (name, clf, Xtr, Xte) in enumerate(model_specs):
    pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('clf', clf)])
    pipe.fit(Xtr, y_train)
    y_prob = pipe.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, color=colors[idx], lw=2, label=f'{name} (AUC={auc(fpr, tpr):.4f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4. Efficiency Analysis

The key insight: SGPO v2 trades a tiny amount of raw AUC for a massive reduction in model complexity.

In [ ]:
# AUC per feature
for m in models:
    eff = m['auc_mean'] / m['n_features']
    print(f"{m['model']:<28} AUC={m['auc_mean']:.4f}  Feat={m['n_features']:>2}  AUC/feat={eff:.4f}")

print(f"\nSGPO v2 achieves {models[-1]['auc_mean']/models[2]['auc_mean']*100:.1f}% of XGBoost's AUC with {models[-1]['n_features']/models[2]['n_features']*100:.0f}% of features.")

## 5. Summary

| Model | Features | AUC-ROC | Sensitivity |
|---|---|---|---|
| Logistic Regression | 42 | 0.9487 | 0.8742 |
| Random Forest | 42 | 0.9563 | 0.8910 |
| XGBoost | 42 | 0.9595 | 0.8971 |
| **SGPO v2** | **8** | **0.9537** | **0.8902** |

SGPO v2 is the most efficient model: near-best performance with minimal features.